## Import essential python packages

In [ ]:
# import spacec first
import spacec as sp
import glob
import numpy as np
import pandas as pd

#import standard packages
import os
import warnings
import matplotlib.pyplot as plt
import scanpy as sc

import anndata as ad
warnings.filterwarnings('ignore')

import fcsparser


import matplotlib.backends.backend_pdf
plt.rc('axes', grid=False)  # remove gridlines
sc.settings.set_figure_params(dpi=80, facecolor='white') # set dpi and background color for scanpy figures

## Initialize variables

In [ ]:
# Specify the path to the data
root_path = "~/CODEX_08262025"
data_filename = root_path + "/celldata_wLinearize.xlsx" # where the data is stored

# where you want to store the output
output_dir = root_path + '/Analysis/Pre_asinh_AfterExclude' # inset your own path
os.makedirs(output_dir, exist_ok=True)

## Load cell data 

In [ ]:
celldata = pd.read_excel(data_filename)
celldata

## Rename essential columns

In [ ]:
celldata.rename({'x': 'X', 
                 'y': 'Y', 
                 'size': 'cell_size',
                 'Code': 'unique_region'}, axis = 'columns', inplace = True)
celldata.columns

## Get indexes of columns of start and end proteins in cell data

In [ ]:
start_idx = list(np.where(celldata.columns.values == "CD45")[0])[0]
end_idx = list(np.where(celldata.columns.values == "Ki67")[0])[0]

print("%d %d" %(start_idx, end_idx))

In [ ]:
keep_cols = np.concatenate([celldata.columns[0:start_idx].values, 
                            celldata.columns[end_idx+1:].values])
#keep_cols

marker_cols = celldata.columns[start_idx:end_idx+1].values

new_celldata = celldata[np.concatenate([marker_cols, keep_cols])]
new_celldata.columns

## Normalization 
### method: ArcSin, zscore(default), double_zscore, and MinMax

In [ ]:
dfz = pd.DataFrame()
## Normalization
for region in new_celldata.unique_region.unique():
    df_reg = new_celldata[new_celldata.unique_region == region]
    df_reg_norm = sp.pp.format(
        data = df_reg,
        list_out = [],
        list_keep = keep_cols,
        method = "ArcSin")
    dfz = pd.concat([dfz, df_reg_norm], axis = 0)
    
'''

## No normalization
for region in new_celldata.unique_region.unique():
    df_reg = new_celldata[new_celldata.unique_region == region]
    dfz = pd.concat([dfz, df_reg], axis = 0)
    
'''

In [ ]:
# get the column index for the last marker 
marker_last_idx = list(np.where(new_celldata.columns.values == "Ki67")[0])[0]
print(marker_last_idx)

# inspect which markers work, and drop the ones that did not work from the clustering step
# make an anndata to be compatible with the downstream clustering step
adata = sp.hf.make_anndata(
    df_nn = dfz,
    col_sum = marker_last_idx, # this is the column index that has the last protein feature # the rest will go into obs
    nonFuncAb_list = [] # Remove the antibodies that are not working from the clustering step
)

### clustering

In [ ]:
clustering_random_seed = 0

In [ ]:
# clustering
adata = sp.tl.clustering(
    adata, 
    clustering='leiden', # can choose between leiden and louvian
    n_neighbors=10, # number of neighbors for the knn graph
    resolution = 1, #clustering resolution (higher resolution gives more clusters)
    reclustering = False, # if true, no computing the neighbors
    marker_list = None, #if it is None, all variable names are used for clustering
    seed=clustering_random_seed, # random seed for clustering - reproducibility
)

In [ ]:
adata.obs.condition = adata.obs.condition.astype("category")
adata.obs.TMA = adata.obs.TMA.astype("category")

In [ ]:
sc.pl.umap(adata, color = ['leiden_1', 'condition'], wspace=0.5) 

In [ ]:
sc.pl.umap(adata, color = ['leiden_1', 'TMA'], wspace=0.5) 

In [ ]:
sc.pl.dotplot(adata, 
              adata.var_names, # The list of markers to show on the x-axis
              'leiden_1', # The cluster column
              dendrogram = True) # Show the dendrogram

### Save Data to files

In [ ]:
adata.write_h5ad(output_dir + "/adata.h5ad")

### Export Expression data

In [ ]:
expr_data = pd.DataFrame(data=np.transpose(adata.X), index=adata.var_names, columns=adata.obs.index.values)
expr_data.to_csv(output_dir + "/expr_data.csv")

### Export UMAP coordinates

In [ ]:
umap_df = pd.DataFrame(adata.obsm['X_umap'], index=adata.obs.index.values, columns=['umap1', 'umap2'])
umap_df.to_csv(output_dir + "/umap.csv")

### Export Cell meta data

In [ ]:
adata.obs.to_csv(output_dir + "/cellmetadata.csv")